In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Loading data

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

# ── Load raw data ─────────────────────────────────────────────────────────────
df_happiness_raw = pd.read_csv('data/world_happiness_combined.csv')
df_population_raw = pd.read_csv('data/Population_development.csv')

print(f"Happiness raw shape:   {df_happiness_raw.shape}")
print(f"Population raw shape:  {df_population_raw.shape}")

Happiness raw shape:   (1231, 12)
Population raw shape:  (266, 71)


In [4]:
# ── Country name mapping (happiness → World Bank names) ──────────────────────
name_map = {
    'Congo':                      'Congo, Rep.',
    'Congo (Brazzaville)':        'Congo, Rep.',
    'Congo (Kinshasa)':           'Congo, Dem. Rep.',
    'Czech Republic':             'Czechia',
    'Egypt':                      'Egypt, Arab Rep.',
    'Eswatini, Kingdom of':       'Eswatini',
    'Gambia':                     'Gambia, The',
    'Hong Kong':                  'Hong Kong SAR, China',
    'Hong Kong S.A.R. of China':  'Hong Kong SAR, China',
    'Hong Kong S.A.R., China':    'Hong Kong SAR, China',
    'Iran':                       'Iran, Islamic Rep.',
    'Ivory Coast':                "Cote d'Ivoire",
    'Kyrgyzstan':                 'Kyrgyz Republic',
    'Laos':                       'Lao PDR',
    'Macedonia':                  'North Macedonia',
    'Russia':                     'Russian Federation',
    'Slovakia':                   'Slovak Republic',
    'Somalia':                    'Somalia, Fed. Rep.',
    'Somaliland Region':          'Somalia, Fed. Rep.',
    'Somaliland region':          'Somalia, Fed. Rep.',
    'South Korea':                'Korea, Rep.',
    'Swaziland':                  'Eswatini',
    'Syria':                      'Syrian Arab Republic',
    'Taiwan':                     None,   # ikke i World Bank data
    'Taiwan Province of China':   None,
    'Trinidad & Tobago':          'Trinidad and Tobago',
    'Turkey':                     'Turkiye',
    'Venezuela':                  'Venezuela, RB',
    'Vietnam':                    'Viet Nam',
    'Palestinian Territories':    'West Bank and Gaza',
    'North Cyprus':               None,
    'Northern Cyprus':            None,
    'Puerto Rico':                'Puerto Rico (US)',
    'Yemen':                      'Yemen, Rep.',
    'Kosovo':                     'Kosovo',
}

df_h = df_happiness_raw.copy()
df_h['Country_matched'] = df_h['Country'].apply(lambda x: name_map.get(x, x))

# Fjern lande uden match (Taiwan, Nordcypern, etc.)
df_h = df_h[df_h['Country_matched'].notna() & (df_h['Country_matched'] != 'xx')].copy()

# Brug hyppigst forekommende Region per land (inkonsistente navne på tværs af år)
region_mode = (df_h.groupby('Country_matched')['Region']
               .agg(lambda x: x.mode()[0]).reset_index()
               .rename(columns={'Country_matched': 'Country_matched', 'Region': 'Region'}))
df_h = df_h.drop(columns='Region').merge(region_mode, on='Country_matched')

print(f"Lande efter navne-matching og rensning: {df_h['Country_matched'].nunique()}")

Lande efter navne-matching og rensning: 162


# Happiness Data

In [5]:
# ── DataFrame 1: Happiness gennemsnit per land ───────────────────────────────
feature_cols = [
    'Happiness Score', 'GDP per Capita', 'Social Support',
    'Healthy Life Expectancy', 'Freedom',
    'Perceptions of Corruption', 'Generosity', 'Dystopia Residual'
]

df_happiness_avg = (
    df_h.groupby(['Country_matched', 'Region'])[feature_cols]
    .mean()
    .reset_index()
    .rename(columns={'Country_matched': 'Country'})
    .round(4)
)

print(f"Shape: {df_happiness_avg.shape}  →  {df_happiness_avg['Country'].nunique()} lande")
df_happiness_avg.head(5)

Shape: (162, 10)  →  162 lande


,Country,Region,Happiness Score,GDP per Capita,Social Support,Healthy Life Expectancy,Freedom,Perceptions of Corruption,Generosity,Dystopia Residual
0,Afghanistan,South Asia,3.1322,2.2127,0.3728,13.3307,0.1710,0.2691,0.1537,1.8190
1,Albania,Central and Eastern Europe,4.8452,3.1338,0.7236,17.7956,0.4922,0.2552,0.1131,1.8211
2,Algeria,Middle East and North Africa,5.4190,3.1635,0.9933,16.9848,0.2481,0.2960,0.0330,2.3726
3,Angola,Sub-Saharan Africa,3.8723,0.7984,0.9384,0.1339,0.0274,0.0716,0.1053,1.8862
4,Argentina,Latin America and Caribbean,6.2710,3.3650,1.1944,17.7838,0.5778,0.2603,0.0207,2.5007


In [6]:
# Basal statistik for df_happiness_avg
print("Basal statistik — df_happiness_avg")
print("=" * 60)
df_happiness_avg[feature_cols].describe().round(3)

Basal statistik — df_happiness_avg


,Happiness Score,GDP per Capita,Social Support,Healthy Life Expectancy,Freedom,Perceptions of Corruption,Generosity,Dystopia Residual
count,162.000,162.000,162.000,162.000,162.000,162.000,162.000,162.000
mean,5.403,2.982,0.983,15.946,0.512,0.277,0.156,2.094
std,1.099,0.977,0.240,6.339,0.144,0.068,0.119,0.477
min,3.132,0.078,0.064,0.134,0.027,0.072,-0.072,0.650
25%,4.533,2.526,0.847,15.020,0.427,0.249,0.064,1.809
50%,5.378,3.127,1.004,17.051,0.535,0.270,0.149,2.141
75%,6.181,3.554,1.185,17.836,0.609,0.297,0.220,2.430
max,7.645,9.672,1.343,70.600,0.854,0.825,0.642,3.019


# Population data

In [7]:
# ── DataFrame 2: Befolkningsudvikling per land (1980–2022) ───────────────────
year_cols = [str(y) for y in range(1980, 2023)]

# Filtrér aggregerede regioner fra (kun individuelle lande)
region_keywords = [
    'income', ' & ', 'Africa Eastern', 'Africa Western',
    'World', 'OECD', 'IDA', 'IBRD', 'Arab World', 'dividend',
    'states', 'Not classified', 'Heavily', 'Fragile', 'Euro area',
    'Sub-Saharan', 'North America', 'South Asia', 'East Asia',
    'Latin America', 'Middle East', 'Central Europe', 'Baltics',
    'Pacific island', 'Caribbean small', 'Other small'
]

def is_region(name):
    return any(kw in name for kw in region_keywords)

df_population = (
    df_population_raw[~df_population_raw['Country Name'].apply(is_region)]
    [['Country Name'] + year_cols]
    .copy()
    .rename(columns={'Country Name': 'Country'})
)
df_population[year_cols] = df_population[year_cols].apply(pd.to_numeric, errors='coerce')

print(f"Shape: {df_population.shape}  →  {df_population['Country'].nunique()} lande, {len(year_cols)} år (1980–2022)")
df_population[['Country', '1980', '1990', '2000', '2010', '2022']].head(5)

Shape: (219, 44)  →  219 lande, 43 år (1980–2022)


,Country,1980,1990,2000,2010,2022
0,Aruba,59909.0,62753.0,90588.0,101838.0,107310.0
2,Afghanistan,13169311.0,12045660.0,20130327.0,28284089.0,40578842.0
4,Angola,8133872.0,11626360.0,16194869.0,23294825.0,35635029.0
5,Albania,2671997.0,3286542.0,3089027.0,2913021.0,2451636.0
6,Andorra,35782.0,52597.0,65685.0,80706.0,79705.0


In [8]:
# Basal statistik for df_population (udvalgte år)
print("Basal statistik — df_population (udvalgte år)")
print("=" * 60)
df_population[['1980', '1990', '2000', '2010', '2022']].describe().map(lambda x: f"{x:,.0f}")

Basal statistik — df_population (udvalgte år)


,1980,1990,2000,2010,2022
count,218,219,219,219,219
mean,"23,994,918","28,398,113","33,071,605","37,787,968","43,598,846"
std,"91,924,168","108,615,708","126,039,238","142,110,045","161,557,150"
min,"7,366","8,798","9,544","10,043","9,992"
25%,"360,271","494,536","621,266","725,444","827,912"
50%,"3,672,286","4,391,236","5,140,037","5,737,971","6,730,654"
75%,"11,465,468","13,265,856","17,168,446","22,332,880","26,980,808"
max,"981,235,000","1,135,185,000","1,262,645,000","1,337,705,000","1,425,423,212"


# Combineret dataframe

In [9]:
# ── DataFrame 3: Kombineret datasæt ─────────────────────────────────────────
df_combined = pd.merge(df_happiness_avg, df_population, on='Country', how='inner')

# Hjælpekolonner til analyse og visualisering
df_combined['Pop_2022']       = df_combined['2022']
df_combined['Pop_1980']       = df_combined['1980']
df_combined['Pop_growth_pct'] = ((df_combined['2022'] - df_combined['1980']) / df_combined['1980'] * 100).round(2)
df_combined['Log_Pop_2022']   = np.log10(df_combined['2022']).round(4)

print(f"Shape: {df_combined.shape}  →  {df_combined['Country'].nunique()} lande")
df_combined[['Country', 'Region', 'Happiness Score', 'Pop_2022', 'Pop_growth_pct', 'Log_Pop_2022']].head(8)

Shape: (162, 57)  →  162 lande


,Country,Region,Happiness Score,Pop_2022,Pop_growth_pct,Log_Pop_2022
0,Afghanistan,South Asia,3.1322,40578842.0,208.13,7.6083
1,Albania,Central and Eastern Europe,4.8452,2451636.0,-8.25,6.3895
2,Algeria,Middle East and North Africa,5.4190,45477389.0,144.41,7.6578
3,Angola,Sub-Saharan Africa,3.8723,35635029.0,338.11,7.5519
4,Argentina,Latin America and Caribbean,6.2710,45407904.0,62.10,7.6571
5,Armenia,Central and Eastern Europe,4.6656,2969200.0,-4.45,6.4726
6,Australia,Australia and New Zealand,7.2436,26018721.0,77.09,7.4153
7,Austria,Western Europe,7.1794,9041851.0,19.77,6.9563


In [10]:
# Basal statistik for df_combined
print("Basal statistik — df_combined")
print("=" * 60)
stat_cols = ['Happiness Score', 'GDP per Capita', 'Social Support', 'Freedom',
             'Pop_2022', 'Pop_growth_pct', 'Log_Pop_2022']
df_combined[stat_cols].describe().round(3)

Basal statistik — df_combined


,Happiness Score,GDP per Capita,Social Support,Freedom,Pop_2022,Pop_growth_pct,Log_Pop_2022
count,162.000,162.000,162.000,162.000,1.620000e+02,161.000,162.000
mean,5.403,2.982,0.983,0.512,4.877310e+07,123.372,7.099
std,1.099,0.977,0.240,0.144,1.621413e+08,129.792,0.682
min,3.132,0.078,0.064,0.027,3.820030e+05,-27.040,5.582
25%,4.533,2.526,0.847,0.427,4.917631e+06,30.690,6.692
50%,5.378,3.127,1.004,0.535,1.124350e+07,96.030,7.051
75%,6.181,3.554,1.185,0.609,3.652507e+07,189.490,7.563
max,7.645,9.672,1.343,0.854,1.425423e+09,891.920,9.154


In [11]:
# Fordeling af lande per region
print("Antal lande per region:")
df_combined['Region'].value_counts().to_frame('Count')

Antal lande per region:


,Count
Region,
Sub-Saharan Africa,43
Central and Eastern Europe,29
Latin America and Caribbean,24
Western Europe,20
Middle East and North Africa,17
Southeast Asia,9
South Asia,7
East Asia,5
Middle East and Northern Africa,3
